# Train segmentation vùng giả mạo trên Kaggle

Trước khi chạy (panel bên phải, mục **Settings**):
- **Accelerator**: GPU P100 (hoặc GPU T4 x2, code chỉ dùng 1 GPU)
- **Internet**: On (để clone code và tải trọng số ImageNet)
- **Input**: Add Input → Your Datasets → dataset ảnh mặt FF++ đã tải lên (private)

Sửa `CONFIG` ở ô dưới, rồi bấm **Save Version → Save & Run All (Commit)** để chạy nền
(tắt trình duyệt vẫn chạy, tối đa 12 giờ). Kết quả nằm ở tab **Output**: `runs/<tên run>/`.

In [ ]:
CONFIG = 'configs/unet.yaml'   # hoặc configs/deeplabv3plus.yaml, configs/segformer.yaml
EXTRA = ''                     # ví dụ: '--epochs 1 --fraction 0.05' để chạy thử

In [ ]:
import glob, os, zipfile

# Kaggle thường tự giải nén file zip khi tạo dataset; nếu không thì giải nén vào /tmp
found = glob.glob('/kaggle/input/**/index/train.csv', recursive=True)
if not found:
    z = glob.glob('/kaggle/input/**/*.zip', recursive=True)[0]
    zipfile.ZipFile(z).extractall('/tmp/data')
    found = ['/tmp/data/index/train.csv']
INDEX = os.path.dirname(found[0])
FACES = os.path.join(os.path.dirname(INDEX), 'faces')
print(INDEX, FACES, os.path.isdir(FACES))
os.environ['TQDM_MININTERVAL'] = '60'   # log thanh tiến trình mỗi phút, tránh log quá dài

In [ ]:
!nvidia-smi
!rm -rf /tmp/code && git clone -q https://github.com/nguyenchitu2005-jpg/DoAnChuyenNganh.git /tmp/code
!pip install -q segmentation-models-pytorch==0.5.0 albumentations==2.0.8

In [ ]:
%cd /tmp/code
!python src/train.py --config {CONFIG} --data-root {FACES} --index-dir {INDEX} --num-workers 4 --runs-dir /kaggle/working/runs {EXTRA}

In [ ]:
# Đánh giá luôn trên FF++ test (DFD và c40 đánh giá ở máy nhà)
for d in glob.glob('/kaggle/working/runs/*'):
    if os.path.exists(d + '/best.pt'):
        !python src/evaluate.py {d}/best.pt --split test